# Interaction graphs of Chevalier et al. reference clusters

Our 50 sampled Boolean networks did **not show clear clustering**, so we do not create cluster-specific graphs for our models here.

Instead, we use the **original cluster interaction-graph analysis from Chevalier et al.** on **their models only**, to reproduce the reference graphs and demonstrate that their analysis works.

The notebook automatically analyzes all three published reference groups: **red (`r`)**, **orange (`y`)**, and **green (`g`)**. It uses the cluster assignments saved by the previous ensemble notebook, without reclustering.

The reference networks are read from `../Data/models/`, and all graphs are saved to `../Results/Ensemble_analysis_outputs/Their_models/`.

## 1. Load packages

Import the packages needed to read Boolean networks and draw interaction graphs.

In [ ]:
from pathlib import Path
import mpbn
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import graphviz
from typing import List, Dict

## 2. Load the three reference clusters

Read the previously saved cluster labels and load each cluster’s `.bnet` files. This uses **only their models** and does not perform new clustering.

In [ ]:
# Input and output locations.
MODEL_DIR = Path("../Data/models")
OUTPUT_DIR = Path("../Results/Ensemble_analysis_outputs/Their_models")
MEMBERSHIP_CSV = OUTPUT_DIR / "hierarchical_cluster_membership.csv"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Use all three published reference groups automatically.
CLUSTERS = ("r", "y", "g")
cluster_names = {"r": "red", "y": "orange", "g": "green"}

# The membership file was created by the earlier ensemble-analysis notebook.
if not MEMBERSHIP_CSV.is_file():
    raise FileNotFoundError(
        f"Missing {MEMBERSHIP_CSV}. Run the reference clustering section "
        "of ensemble_analysis_two_ensembles_explained.ipynb first."
    )

membership = pd.read_csv(MEMBERSHIP_CSV)
if not {"model", "reference_cluster"}.issubset(membership.columns):
    raise ValueError("Cluster CSV must contain 'model' and 'reference_cluster'.")

# Keep each cluster's networks separate for the later steps.
model_files_by_cluster = {}
solutions_by_cluster = {}

for cluster in CLUSTERS:
    # Find model filenames assigned to the current reference group.
    selected_names = membership.loc[
        membership["reference_cluster"] == cluster_names[cluster], "model"
    ].astype(str).tolist()
    if not selected_names:
        raise ValueError(f"No models listed for cluster {cluster_names[cluster]}.")

    modelname = [str(MODEL_DIR / Path(name).name) for name in selected_names]
    missing = [name for name in modelname if not Path(name).is_file()]
    if missing:
        raise FileNotFoundError(f"Missing model file(s): {missing[:5]}")

    # Load the Boolean networks for this reference group.
    solutions = [mpbn.MPBooleanNetwork(filename) for filename in modelname]
    model_files_by_cluster[cluster] = modelname
    solutions_by_cluster[cluster] = solutions

    print(f"Reference cluster: {cluster_names[cluster]}")
    print(f"Loaded {len(solutions)} models from {MODEL_DIR}")

## 3. Extract Boolean functions

For each reference cluster, extract each gene’s rule from every model. The rules are stored separately for red, orange, and green.

In [ ]:
clauses_by_cluster = {}

for cluster in CLUSTERS:
    solutions = solutions_by_cluster[cluster]
    # For each model, I store the clauses per node
    clauses_per_model = list() #Each element of this list will correspond to a model, it's a dict linking nodes and their clauses.
    for i, model in enumerate(solutions): # Go through the models of this cluster.
        clauses_of_a_node = dict() #The dict that will link a node to its clauses.
        for node in model.keys(): #We go through the nodes of the model.
            #if not isinstance(model[node],frozenset):
            if model[node]==True or model[node]==False :
                clauses_of_a_node[node] = model[node]
            else:
                clauses_of_a_node[node] = mpbn.minibn.struct_of_dnf(model.ba, model[node]) #We store the clauses ruling this node.
        clauses_per_model.append(clauses_of_a_node) #Now that all nodes are linked with their clauses, we save this information as a new element in clauses_per_model.
    clauses_by_cluster[cluster] = clauses_per_model

## 4. Count regulatory influences

Within each cluster, count how often a gene activates or inhibits another gene. Also count constant Boolean rules.

In [ ]:
influences_by_cluster = {}
constants_by_cluster = {}

for cluster in CLUSTERS:
    clauses_per_model = clauses_by_cluster[cluster]
    influences = dict()
    constantes = dict()
    for model in clauses_per_model:
        for node, influenceurs_set in model.items():
            temp = set()
            if node not in influences:
                influences[node] = dict()
            #print(influenceurs_set)
            #print(type(influenceurs_set))
            #print(isinstance(influenceurs_set, bool))
            if influenceurs_set != True and influenceurs_set != False:
                for conjonctions in influenceurs_set:
                    for clause in conjonctions:
                        if clause[0] not in temp:
                            temp.add(clause[0])
                            if clause[0] not in influences[node]:
                                influences[node][clause[0]] = dict()
                                influences[node][clause[0]][clause[1]] = 1
                            elif clause[1] not in influences[node][clause[0]]:
                                influences[node][clause[0]][clause[1]] = 1
                            else:
                                influences[node][clause[0]][clause[1]] += 1
            else:
                if node not in constantes:
                    constantes[node] = dict()
                if influenceurs_set not in constantes[node]:
                    constantes[node][influenceurs_set] = 1
                else:
                    constantes[node][influenceurs_set] += 1
    influences_by_cluster[cluster] = influences
    constants_by_cluster[cluster] = constantes

## 5. Set up edge widths

This helper scales the drawn arrow widths based on how often each interaction appears. The same function is used for all three groups.

In [ ]:
from typing import List, Dict


def get_arcs_size(l_input: List, min_intensity: int = 1, max_intensity: int = 10) -> Dict[int, str]:
    res = {}
    get_arc_size_rec(l_input, res, min_intensity, max_intensity)
    return res
       
def get_arc_size_rec(l_rec: List, res: Dict[int, str], inf: int, sup: int, left: bool = False):
    max_diff = 0
    index_max_diff = 0
    for i in range(len(l_rec) - 1):
        diff = l_rec[i + 1] - l_rec[i]
        if diff > max_diff:
            max_diff = diff
            index_max_diff = i

    if max_diff == 0 or inf == sup:
        for element in l_rec:
            if left:
                res[element] = str(inf)
            else:
                res[element] = str(sup)
    else:
        get_arc_size_rec(l_rec[0:index_max_diff+1], res, inf, (inf+sup)//2, True)
        get_arc_size_rec(l_rec[index_max_diff+1:], res, (inf+sup)//2 + 1, sup)

## 6. Count edge frequencies

Group the activating and inhibiting influences by their frequency within each cluster.

In [ ]:
occurrences_by_cluster = {}
intensity_by_cluster = {}

for cluster in CLUSTERS:
    influences = influences_by_cluster[cluster]
    occurrences = dict()
    for target, sources in influences.items():
        for source, infl in sources.items():
            for influence, occurrence in infl.items():
                if occurrence not in occurrences:
                    occurrences[occurrence] = list()
                occurrences[occurrence].append((source,target))

    max_intensity = 8
    intensity = get_arcs_size(sorted(occurrences.keys()), max_intensity=max_intensity)
    intensity[len(intensity)+1]=str(max_intensity+5)
    occurrences_by_cluster[cluster] = occurrences
    intensity_by_cluster[cluster] = intensity

## 7. Show edge-count summaries

Print how many regulatory interactions occur in different numbers of models in each group.

In [ ]:
# Show the original edge-frequency summary for each reference group.
for cluster in CLUSTERS:
    print(f"\n{cluster_names[cluster].capitalize()} cluster:")
    occurrences = occurrences_by_cluster[cluster]
    for occurrence, arcs in sorted(occurrences.items()):
        print(f"{len(arcs):2d} edges present in {occurrence:3d} models")

## 8. Create three Graphviz graphs

Create one empty influence graph for each reference cluster.

In [ ]:
graphs_by_cluster = {}

for cluster in CLUSTERS:
    models_per_edge = graphviz.Digraph('models_per_edge', comment='Display the number of models having each edge in its influence graph.')
    models_per_edge.graph_attr['ratio'] = '1.1'
    graphs_by_cluster[cluster] = models_per_edge

## 9. Count different Boolean functions per gene

For each gene, count the number of distinct Boolean rules across models in the same group.

In [ ]:
different_functions_by_cluster = {}
function_counts_by_cluster = {}

for cluster in CLUSTERS:
    influences = influences_by_cluster[cluster]
    clauses_per_model = clauses_by_cluster[cluster]
    fonctions_differentes = dict()
    fonctions_differentes_avec_occurrence = dict()

    for node in influences.keys():
        fonctions_differentes[node] = set()
        fonctions_differentes_avec_occurrence[node] = dict()
        for model in clauses_per_model:
            fonctions_differentes[node].add(model[node])
            if model[node] not in fonctions_differentes_avec_occurrence[node]:
                fonctions_differentes_avec_occurrence[node][model[node]] = 1
            else:
                fonctions_differentes_avec_occurrence[node][model[node]] += 1
    different_functions_by_cluster[cluster] = fonctions_differentes
    function_counts_by_cluster[cluster] = fonctions_differentes_avec_occurrence

## 10. List invariant genes

List genes whose Boolean rule is the same in every model of each reference group.

In [ ]:
# Invariant genes are determined separately for every reference cluster.
for cluster in CLUSTERS:
    fonctions_differentes = different_functions_by_cluster[cluster]
    noeuds_communs_a_tout_modele = []
    for node, functions in fonctions_differentes.items():
        if len(functions) == 1:
            noeuds_communs_a_tout_modele.append(node)

    print(f"{cluster_names[cluster].capitalize()} cluster - invariant genes:")
    print(noeuds_communs_a_tout_modele)

## 11. Draw genes without constant rules

For each cluster, draw gene nodes and color them by the number of distinct Boolean functions.

In [ ]:
for cluster in CLUSTERS:
    models_per_edge = graphs_by_cluster[cluster]
    fonctions_differentes = different_functions_by_cluster[cluster]
    constantes = constants_by_cluster[cluster]
    # Graph: creation of non-constant nodes:

    for node in fonctions_differentes:
        if node not in constantes:
            if len(fonctions_differentes[node]) == 1:
                models_per_edge.node(node, label=f"{node}\n{len(fonctions_differentes[node])} func", fillcolor="darkgoldenrod2", style="rounded,filled,bold", shape="oval", fontcolor="black", fontname="arial bold", fontsize="50pt")
            elif len(fonctions_differentes[node]) == 2:
                models_per_edge.node(node, label=f"{node}\n{len(fonctions_differentes[node])} func", fillcolor="lightgoldenrod1" , style="rounded,filled", shape="oval", fontsize="40pt")
            elif len(fonctions_differentes[node]) == 3:
                models_per_edge.node(node, label=f"{node}\n{len(fonctions_differentes[node])} func", fillcolor="cornsilk" , style="rounded,filled", shape="oval", fontsize="35pt")
            elif len(fonctions_differentes[node]) < 10:
                models_per_edge.node(node, label=f"{node}\n{len(fonctions_differentes[node])} func", fillcolor="white" , style="rounded,filled", shape="oval", fontsize="30pt")
            else:
                models_per_edge.node(node, label=f"{node}\n{len(fonctions_differentes[node])} func", fillcolor="white", style="rounded,filled,dotted", shape="oval", fontsize="30pt")

## 12. Draw genes with constant rules

Add rectangular nodes for genes assigned a constant true or false function in the cluster.

In [ ]:
for cluster in CLUSTERS:
    models_per_edge = graphs_by_cluster[cluster]
    fonctions_differentes = different_functions_by_cluster[cluster]
    constantes = constants_by_cluster[cluster]
    # Graph: creation of constant nodes:

    for constante, influence in constantes.items():
        for signe, occurrence in influence.items():
            #signe = 1 if signe == True else 0
            if len(fonctions_differentes[constante]) == 1:
                models_per_edge.node(constante, label=f"{constante}\n{len(fonctions_differentes[constante])} func\n{signe}: {occurrence}", fillcolor="darkgoldenrod2", style="rounded,filled,bold", shape="box", fontcolor="black", fontname="arial bold", fontsize="45pt")
            elif len(fonctions_differentes[constante]) == 2:
                models_per_edge.node(constante, label=f"{constante}\n{len(fonctions_differentes[constante])} func\n{signe}: {occurrence}", fillcolor="lightgoldenrod1", style="rounded,filled", shape="box", fontsize="35pt")
            elif len(fonctions_differentes[constante]) == 3:
                models_per_edge.node(constante, label=f"{constante}\n{len(fonctions_differentes[constante])} func\n{signe}: {occurrence}", fillcolor="cornsilk", style="rounded,filled", shape="box", fontsize="30pt")
            elif len(fonctions_differentes[constante]) < 10:
                models_per_edge.node(constante, label=f"{constante}\n{len(fonctions_differentes[constante])} func\n{signe}: {occurrence}", fillcolor="white", style="rounded,filled", shape="box", fontsize="25pt")
            else:
                models_per_edge.node(constante, label=f"{constante}\n{len(fonctions_differentes[constante])} func\n{signe}: {occurrence}", fillcolor="white", style="rounded,filled,dotted", shape="box", fontsize="25pt")

## 13. Draw regulatory influences

Add activating (green) and inhibiting (red) interactions to each cluster’s graph. Labels show how many models contain each influence.

In [ ]:
for cluster in CLUSTERS:
    models_per_edge = graphs_by_cluster[cluster]
    influences = influences_by_cluster[cluster]
    occurrences = occurrences_by_cluster[cluster]
    intensity = intensity_by_cluster[cluster]
    modelname = model_files_by_cluster[cluster]
    # Graph: creation of the arcs, colored according to the activating/inhibiting influence:

    occurrences_liste = list()
    for target, sources in influences.items():
        #if target=="STAT3":
        for source, infl in sources.items():
            #print(f"{source}, {infl}")
            for influence, occurrence in infl.items():
                occurrences_liste.append(occurrence)
                if influence == True:
                    if (source, target) in occurrences.get(len(modelname), []):
                        models_per_edge.edge(source, target, label=f"{occurrence}", penwidth=intensity[occurrence], color="darkgreen", fontcolor="darkgreen", fontname="arial bold", fontsize="30pt", arrowsize="2")
                    else:
                        # Alternative edge styling without the scaled width (unused).
                        #models_per_edge.edge(source, target, label=str(occurrence), penwidth=str(occurrence/100), color="green")
                        models_per_edge.edge(source, target, label=str(occurrence), penwidth=intensity[occurrence], color="darkolivegreen3", fontcolor="darkolivegreen3", fontname="arial bold", fontsize="30pt", arrowsize="2")
                else:
                    if (source, target) in occurrences.get(len(modelname), []):
                        models_per_edge.edge(source, target, label=str(occurrence), arrowhead="tee", penwidth=intensity[occurrence], color="firebrick", fontcolor="firebrick", fontname="arial bold", fontsize="30pt", arrowsize="3")
                    else:
                        #models_per_edge.edge(source, target, label=str(occurrence), arrowhead="tee", penwidth=str(occurrence/100), color="red"
                        models_per_edge.edge(source, target, label=str(occurrence), arrowhead="tee", penwidth=intensity[occurrence], color="lightcoral", fontcolor="lightcoral", fontname="arial bold", fontsize="30pt", arrowsize="3")
                #print(f"{source} {target} {influence} {occurrence}")

## 14. Save all three graphs

Render separate PDFs and Graphviz source files for **r**, **y**, and **g**. No manual cluster selection or repeated notebook execution is needed.

In [ ]:
for cluster in CLUSTERS:
    models_per_edge = graphs_by_cluster[cluster]
    # Save this reference cluster graph; do not open an external viewer.
    models_per_edge.render(
        filename=f"IGstat-cluster-{cluster}",
        directory=str(OUTPUT_DIR),
        format="pdf",
        view=False,
    )
    print(f"Saved: {OUTPUT_DIR / f'IGstat-cluster-{cluster}.pdf'}")